## What is Autograd ?
Autograd is a core component of PyTorch that provide automatic differentiation for tensor operations. It enables gradient computation , which is essential for training machine learning models using optimization algorithms like *gradient descent*.

In simple terms, Autograd automatically calculates the derivatives of complex, nested operations (often represented as computational graphs) on tensors. This automatic differentiation is crucial for training machine learning models, especially neural networks, as it efficiently provides the gradients needed for optimization algorithms like gradient descent.


> It is mainly useful while calculating the gradients in backward pass(Propogation)

In [1]:
import torch

Lets take some examples to calculate the Derivatives of the equation :

#### ***Example-1 : A SIngle Equation***

Equation :

> y = x^2

Derivative :
> dy/dx = 2*x


In [4]:
x = torch.tensor(3.0,requires_grad=True) # Here we are setting the requires_grad = True, because it will helpful while performing the further derivatives operations
x

tensor(3., requires_grad=True)

In [6]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [ ]:
# Now getting the dy_dx
y.backward()
dy_dx = x.grad

In [10]:
dy_dx

tensor(6.)

#### ***Example-2 : Nested Equation***

Equations :

> y = x^2 ,
> z = sin(y)

Derivatives :
> dz/dy = cos(y) ,
> dy/dx = 2*X

> So final dz/dx = dz/dy * dy/dx

> dz/dx = 2 * X * cos(y)


In [19]:
x = torch.tensor(2.0,requires_grad=True)
x

tensor(2., requires_grad=True)

In [20]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [21]:
z = torch.sin(y)
z

tensor(-0.7568, grad_fn=<SinBackward0>)

In [22]:
# Now Calculating derivatives
z.backward()

In [23]:
# So here to get dz/dx, we dont need to calculate the dz/dy and dy/dx .
# Rather PyTorch autograd and by remembering the previous operation of Forward propogation, it will get the direct dz/dx value

dz_dx = x.grad
dz_dx # Here dz_dx = 2*x*cos(y)

tensor(-2.6146)

#### ***Example-3 : A Simple Neural Network***

Equations :



```
1. Linear Transformation :
   z = w*x + b

2. Activation (Sigmoid) Function :
  y_pred = sigmoid(z) = 1/(1 + e^-z)

3. Loss Function(Binary Cross-Entropy Loss) :
  L = [y_target * ln(y_pred)] + (1-y_target) * ln(1 - y_pred)]
```



Derivatives :
```
1.
  dL/dw = dL/dY_pred * dY_pred/dz * dz/dw

  So final :
  dL/dw = (y_pred - y) * x

2.
  dl/db = dL/dY_pred * dY_pred/dz * dz/db

  So final :
  dL/db = (y_pred - y) * 1

```


Calculating the derivative for this neural network using ***manual way***

In [27]:
import torch

x = torch.tensor(6.7) # Input Feature
y = torch.tensor(0.0) # Target Label (binary 0 or 1)


w = torch.tensor(1.0) # Initial weight
b = torch.tensor(0.0) # Initial Bias


# ===================== Forward Pass (Propogation) ========================

# Sigmoid activation function
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

# Forward pass
z = w * x + b
y_pred = sigmoid(z)

# Binary Cross-Entropy Loss (BCE Loss)
# Note: The formula provided in the markdown was missing a negative sign for standard BCE loss.
# Using the correct BCE loss formula here.
loss = - (y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))

print(f"Predicted y_pred: {y_pred.item():.4f}")
print(f"Calculated Loss (L): {loss.item():.4f}")


# ===================== Backward Pass (Propogation) ========================
# Manual derivatives based on the provided formulas
dL_dy_pred = -(y / y_pred - (1 - y) / (1 - y_pred))
dy_pred_dz = y_pred * (1 - y_pred)
dz_dw = x
dz_db = torch.tensor(1.0)

# Apply chain rule for dL/dw and dL/db
# dL/dw = dL/dy_pred * dy_pred/dz * dz/dw
dL_dw = (y_pred - y) * x # This simplifies from the chain rule for this specific setup

# dL/db = dL/dy_pred * dy_pred/dz * dz/db
dL_db = (y_pred - y) * 1 # This simplifies from the chain rule for this specific setup

print(f"Manual derivative dL/dw: {dL_dw.item():.4f}")
print(f"Manual derivative dL/db: {dL_db.item():.4f}")

Predicted y_pred: 0.9988
Calculated Loss (L): 6.7012
Manual derivative dL/dw: 6.6918
Manual derivative dL/db: 0.9988


Calculating the derivative for this neural network using ***PyTorch Autograd***

In [31]:
x = torch.tensor(6.7) # Input Feature
y = torch.tensor(0.0) # Target Label (binary 0 or 1)


w = torch.tensor(1.0,requires_grad=True) # Initial weight
b = torch.tensor(0.0,requires_grad=True) # Initial Bias


# ===================== Forward Pass (Propogation) ========================

# Sigmoid activation function
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

# Forward pass
z = w * x + b
y_pred = sigmoid(z)

# Binary Cross-Entropy Loss (BCE Loss)
# Note: The formula provided in the markdown was missing a negative sign for standard BCE loss.
# Using the correct BCE loss formula here.
loss = - (y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))

print(f"Predicted y_pred: {y_pred.item():.4f}")
print(f"Calculated Loss (L): {loss.item():.4f}")


Predicted y_pred: 0.9988
Calculated Loss (L): 6.7012


In [32]:
# ===================== Backward Pass (Propogation) ========================
# Here we do not need to manually calculate the derivatives , PyTorch autograd will take care of it
loss.backward()


In [34]:

dloss_dw = w.grad
dloss_db = b.grad
print("Derivate dL/dw : ",dloss_dw)
print("Derivate dL/db : ",dloss_db)

Derivate dL/dw :  tensor(6.6914)
Derivate dL/db :  tensor(0.9987)


#### ***Clearing gradient***

Here we will clear the gradient if we are passing the backward pass in multiple passes, so that it should not accumulate the result of the previous backward pass and we will get the proper gradient for the current backward pass

In [45]:
import torch

# ===== Forward Pass ======
x = torch.tensor(2.0,requires_grad=True)
print("x : ",x)

x :  tensor(2., requires_grad=True)


In [72]:
y = x ** 2
print("y : ",y)

y :  tensor(4., grad_fn=<PowBackward0>)


In [73]:
 # Backward Pass
y.backward()

In [74]:
# Here it is acuumulating the the result/ gradient of the previous backward pass.
# Actual value should be 4.0 , but it is accumulating the result of previous gradient and getting some more value then actual gradient value .
x.grad

tensor(8.)

In [68]:
# So to avoid the accumulating of the gradient / result , we will clear the gradient of the previous pass by using the below method
x.grad.zero_()

tensor(0.)